Research question: Does a recovery method that improves overall F1 also perform consistently across different groups of credit-card clients?

The subgroup comparison uses missingness deliberately injected into `BILL_AMT1` on copies of complete validation features.


In [3]:

# Initialize the SHML environment

from pathlib import Path
import sys

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        path for path in (cwd, *cwd.parents)
        if (path / "src" / "run_pipeline.py").exists()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Open this notebook inside the SHML repository."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation import evaluate_model

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "working"
    / "default of credit card clients.xls"
)
df = pd.read_excel(DATA_PATH, header=1, engine="xlrd")

target = "default payment next month"
X = df.drop(columns=["ID", target])
y = df[target]

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
baseline_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)
baseline_model.fit(X_train_scaled, y_train)

print("SHML initialized successfully.")


SHML initialized successfully.


In [4]:

# Phase G — Step 15A
# Subgroup robustness: Zero vs Median recovery

import numpy as np
import pandas as pd

FEATURE = "BILL_AMT1"
MISSINGNESS = 0.30
SEEDS = range(20)

training_median = X_train[FEATURE].median()

# Predefined groups
age_groups = pd.cut(
    X_val["AGE"],
    bins=[-np.inf, 34, 49, 64, np.inf],
    labels=["Under 35", "35-49", "50-64", "65+"]
)

group_definitions = {
    "SEX_1": X_val["SEX"].eq(1),
    "SEX_2": X_val["SEX"].eq(2),
    "Age_Under_35": age_groups.eq("Under 35"),
    "Age_35_49": age_groups.eq("35-49"),
    "Age_50_64": age_groups.eq("50-64"),
    "Age_65_Plus": age_groups.eq("65+")
}

subgroup_results = []

for seed in SEEDS:

    X_corrupted = X_val.copy(deep=True)
    rng = np.random.default_rng(seed)

    positions = rng.choice(
        len(X_val),
        size=int(len(X_val) * MISSINGNESS),
        replace=False
    )

    X_corrupted.iloc[
        positions,
        X_corrupted.columns.get_loc(FEATURE)
    ] = np.nan

    X_zero = X_corrupted.copy(deep=True)
    X_zero[FEATURE] = X_zero[FEATURE].fillna(0)

    X_median = X_corrupted.copy(deep=True)
    X_median[FEATURE] = X_median[FEATURE].fillna(
        training_median
    )

    for strategy, X_candidate in {
        "Zero Recovery": X_zero,
        "Fixed Median": X_median
    }.items():

        for group_name, group_mask in group_definitions.items():

            X_group = X_candidate.loc[group_mask]
            y_group = y_val.loc[group_mask]

            # Avoid invalid metrics for very small groups
            if len(X_group) < 30 or y_group.nunique() < 2:
                continue

            metrics = evaluate_model(
                X_batch=X_group,
                y_true=y_group,
                model=baseline_model,
                scaler=scaler
            )

            subgroup_results.append({
                "Seed": seed,
                "Group": group_name,
                "Strategy": strategy,
                "N_Observations": len(X_group),
                "N_Positive": int(y_group.sum()),
                "F1": metrics["f1"],
                "Recall": metrics["recall"],
                "ROC_AUC": metrics["roc_auc"]
            })

subgroup_df = pd.DataFrame(subgroup_results)

subgroup_summary = (
    subgroup_df
    .groupby(["Group", "Strategy"])
    .agg(
        N_Seeds=("Seed", "nunique"),
        N_Observations=("N_Observations", "first"),
        N_Positive=("N_Positive", "first"),
        Mean_F1=("F1", "mean"),
        Mean_Recall=("Recall", "mean"),
        Mean_AUC=("ROC_AUC", "mean")
    )
    .reset_index()
)

display(subgroup_summary.round(4))


,Group,Strategy,N_Seeds,N_Observations,N_Positive,Mean_F1,Mean_Recall,Mean_AUC
0,Age_35_49,Fixed Median,20,1756,407,0.3837,0.2615,0.7096
1,Age_35_49,Zero Recovery,20,1756,407,0.3888,0.2671,0.7083
2,Age_50_64,Fixed Median,20,400,95,0.4586,0.3532,0.7375
3,Age_50_64,Zero Recovery,20,400,95,0.4654,0.3600,0.7362
4,Age_Under_35,Fixed Median,20,2327,490,0.3376,0.2267,0.7055
5,Age_Under_35,Zero Recovery,20,2327,490,0.3508,0.2399,0.7046
6,SEX_1,Fixed Median,20,1797,402,0.4039,0.2909,0.7320
7,SEX_1,Zero Recovery,20,1797,402,0.4064,0.2950,0.7311
8,SEX_2,Fixed Median,20,2703,593,0.3426,0.2262,0.6940
9,SEX_2,Zero Recovery,20,2703,593,0.3566,0.2392,0.6930


Finding 1 — Zero recovery improves mean subgroup F1 relative to median. All five reported groups show improvements, supporting the recovery method's usefulness under the fixed F1 objective.

Finding 2 — Substantial subgroup differences remain. For example, under zero recovery, the mean F1 is 0.4064 for SEX = 1 versus 0.3566 for SEX = 2. This difference warrants investigation; it is not by itself proof of unfairness or evidence that recovery caused the gap.

Finding 3 — ROC-AUC decreases slightly under zero recovery in every reported subgroup. This is consistent with the trade-off observed in the overall experiments.

Finding 4 — We have not yet established subgroup safety. Higher F1 than median doesn't establish that a subgroup remains above its healthy performance threshold. We need to compare each subgroup against its own healthy reference, not just another recovery strategy.

In [5]:

# Phase G — Step 15B
# Paired subgroup comparison: Zero minus Median

import numpy as np
import pandas as pd
from scipy import stats

paired_subgroups = subgroup_df.pivot(
    index=["Seed", "Group"],
    columns="Strategy",
    values=["F1", "Recall", "ROC_AUC"]
)

subgroup_paired_results = []

for group in subgroup_df["Group"].unique():

    subset = paired_subgroups.xs(
        group,
        level="Group"
    )

    for metric in ["F1", "Recall", "ROC_AUC"]:

        differences = (
            subset[(metric, "Zero Recovery")]
            - subset[(metric, "Fixed Median")]
        ).to_numpy()

        n = len(differences)
        mean_diff = differences.mean()

        if n > 1 and not np.isclose(
            differences.std(ddof=1), 0
        ):
            margin = (
                stats.t.ppf(0.975, df=n - 1)
                * stats.sem(differences)
            )
        else:
            margin = 0.0

        subgroup_paired_results.append({
            "Group": group,
            "Metric": metric,
            "N_Seeds": n,
            "Mean_Difference": mean_diff,
            "CI_Lower": mean_diff - margin,
            "CI_Upper": mean_diff + margin,
            "Wins": int((differences > 0).sum()),
            "Losses": int((differences < 0).sum()),
            "Ties": int((differences == 0).sum())
        })

subgroup_paired_df = pd.DataFrame(
    subgroup_paired_results
)

display(
    subgroup_paired_df[
        subgroup_paired_df["Metric"] == "F1"
    ].round(6)
)


,Group,Metric,N_Seeds,Mean_Difference,CI_Lower,CI_Upper,Wins,Losses,Ties
0,SEX_1,F1,20,0.002463,0.000485,0.004442,13,7,0
3,SEX_2,F1,20,0.014035,0.011767,0.016303,20,0,0
6,Age_Under_35,F1,20,0.013156,0.010605,0.015707,20,0,0
9,Age_35_49,F1,20,0.005164,0.003513,0.006815,19,1,0
12,Age_50_64,F1,20,0.006813,0.003145,0.010480,10,0,10


In [6]:

# Phase G — Step 15C
# Compare zero-recovery subgroup performance
# with healthy subgroup performance

healthy_subgroups = []

for group_name, group_mask in group_definitions.items():

    X_group = X_val.loc[group_mask]
    y_group = y_val.loc[group_mask]

    if len(X_group) < 30 or y_group.nunique() < 2:
        print(
            "Excluded:", group_name,
            "N =", len(X_group),
            "Positive =", int(y_group.sum())
        )
        continue

    metrics = evaluate_model(
        X_batch=X_group,
        y_true=y_group,
        model=baseline_model,
        scaler=scaler
    )

    healthy_subgroups.append({
        "Group": group_name,
        "Healthy_F1": metrics["f1"],
        "Healthy_Recall": metrics["recall"],
        "Healthy_AUC": metrics["roc_auc"]
    })

healthy_subgroup_df = pd.DataFrame(
    healthy_subgroups
)

zero_summary = subgroup_summary[
    subgroup_summary["Strategy"] == "Zero Recovery"
].copy()

subgroup_safety = zero_summary.merge(
    healthy_subgroup_df,
    on="Group",
    validate="one_to_one"
)

subgroup_safety["F1_Loss_Pct"] = np.where(
    subgroup_safety["Healthy_F1"] > 0,
    100 * (
        subgroup_safety["Healthy_F1"]
        - subgroup_safety["Mean_F1"]
    ) / subgroup_safety["Healthy_F1"],
    np.nan
)

subgroup_safety["Recall_Loss_Pct"] = np.where(
    subgroup_safety["Healthy_Recall"] > 0,
    100 * (
        subgroup_safety["Healthy_Recall"]
        - subgroup_safety["Mean_Recall"]
    ) / subgroup_safety["Healthy_Recall"],
    np.nan
)

subgroup_safety["Safety_Flag"] = np.select(
    [
        subgroup_safety[
            ["F1_Loss_Pct", "Recall_Loss_Pct"]
        ].isna().any(axis=1),
        (
            (subgroup_safety["F1_Loss_Pct"] > 10)
            |
            (subgroup_safety["Recall_Loss_Pct"] > 10)
        )
    ],
    ["INSUFFICIENT", "REVIEW"],
    default="WITHIN_TOLERANCE"
)

display(
    subgroup_safety[
        [
            "Group",
            "N_Observations",
            "Healthy_F1",
            "Mean_F1",
            "F1_Loss_Pct",
            "Recall_Loss_Pct",
            "Safety_Flag"
        ]
    ].round(4)
)


Excluded: Age_65_Plus N = 17 Positive = 3


,Group,N_Observations,Healthy_F1,Mean_F1,F1_Loss_Pct,Recall_Loss_Pct,Safety_Flag
0,Age_35_49,1756,0.3676,0.3888,-5.7618,-8.7000,WITHIN_TOLERANCE
1,Age_50_64,400,0.4476,0.4654,-3.9926,-6.8750,WITHIN_TOLERANCE
2,Age_Under_35,2327,0.3297,0.3508,-6.3808,-9.8598,WITHIN_TOLERANCE
3,SEX_1,1797,0.3908,0.4064,-3.9684,-6.8468,WITHIN_TOLERANCE
4,SEX_2,2703,0.3320,0.3566,-7.3971,-10.8203,WITHIN_TOLERANCE


In [7]:

# Phase G — Step 15D
# Seed-level subgroup safety validation

import numpy as np
import pandas as pd

zero_seed_results = subgroup_df[
    subgroup_df["Strategy"] == "Zero Recovery"
].copy()

seed_safety = zero_seed_results.merge(
    healthy_subgroup_df,
    on="Group",
    validate="many_to_one"
)

seed_safety["F1_Loss_Pct"] = np.where(
    seed_safety["Healthy_F1"] > 0,
    100 * (
        seed_safety["Healthy_F1"] - seed_safety["F1"]
    ) / seed_safety["Healthy_F1"],
    np.nan
)

seed_safety["Recall_Loss_Pct"] = np.where(
    seed_safety["Healthy_Recall"] > 0,
    100 * (
        seed_safety["Healthy_Recall"]
        - seed_safety["Recall"]
    ) / seed_safety["Healthy_Recall"],
    np.nan
)

seed_safety["Decision"] = np.select(
    [
        seed_safety[
            ["F1_Loss_Pct", "Recall_Loss_Pct"]
        ].isna().any(axis=1),
        (
            (seed_safety["F1_Loss_Pct"] > 10)
            |
            (seed_safety["Recall_Loss_Pct"] > 10)
        )
    ],
    ["INSUFFICIENT", "REVIEW"],
    default="WITHIN_TOLERANCE"
)

seed_safety_summary = (
    seed_safety
    .groupby("Group")
    .agg(
        Evaluations=("Seed", "size"),
        Within_Tolerance=(
            "Decision",
            lambda s: (s == "WITHIN_TOLERANCE").sum()
        ),
        Review=(
            "Decision",
            lambda s: (s == "REVIEW").sum()
        ),
        Insufficient=(
            "Decision",
            lambda s: (s == "INSUFFICIENT").sum()
        ),
        Worst_F1_Loss_Pct=("F1_Loss_Pct", "max"),
        Worst_Recall_Loss_Pct=("Recall_Loss_Pct", "max")
    )
    .reset_index()
)

seed_safety_summary["Pass_Rate_Pct"] = (
    100 * seed_safety_summary["Within_Tolerance"]
    / seed_safety_summary["Evaluations"]
)

display(seed_safety_summary.round(4))


,Group,Evaluations,Within_Tolerance,Review,Insufficient,Worst_F1_Loss_Pct,Worst_Recall_Loss_Pct,Pass_Rate_Pct
0,Age_35_49,20,20,0,0,-2.6050,-6.0000,100.0
1,Age_50_64,20,20,0,0,2.0548,0.0000,100.0
2,Age_Under_35,20,20,0,0,-2.3083,-4.6729,100.0
3,SEX_1,20,20,0,0,-0.9499,-3.6036,100.0
4,SEX_2,20,20,0,0,-3.3245,-5.4688,100.0


In [8]:

# Check why age 65+ was excluded

older_mask = group_definitions["Age_65_Plus"]

print("Age 65+ observations:", int(older_mask.sum()))
print("Age 65+ defaults:", int(y_val.loc[older_mask].sum()))
print("Age 65+ unique classes:", y_val.loc[older_mask].nunique())


Age 65+ observations: 17
Age 65+ defaults: 3
Age 65+ unique classes: 2



### Observation G.1 — Subgroup Coverage and Limitations

Subgroup robustness was evaluated under 30% BILL_AMT1
missingness across 20 random injection seeds.

Five subgroups satisfied the predefined evaluation
requirements:

- Age under 35
- Age 35–49
- Age 50–64
- SEX = 1
- SEX = 2

The Age 65+ subgroup was excluded because it contained
only 17 validation observations, including 3 defaults
and 14 non-defaults.

Although both classes were represented, the small
sample size made F1 and recall estimates potentially
unstable.

For the five evaluated groups, zero imputation
produced higher mean F1 and recall than each group's
healthy baseline.

These findings provide preliminary evidence of
subgroup robustness under the tested conditions.

However, results do not establish robustness for
the excluded Age 65+ population.

Seed-level safety analysis is required before
concluding that all evaluated groups consistently
satisfy the 10% F1 and recall tolerances.
